In [ ]:
# 


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier




FEATURE_CSV = "/kaggle/input/extracted-features-dataset/glcm_standardized_features_with_fertility.csv"
TARGET_COL = "fertility"
OUTPUT_DIR = "/kaggle/working/SoilF_ML_results"
SEED =123
N_SPLITS = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)


# LOAD & CLEAN DATA

df = pd.read_csv(FEATURE_CSV)
df = df.drop(columns=["image"], errors="ignore")

# CLEAN LABELS
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip().str.lower()

X = df.drop(columns=[TARGET_COL]).values
y_raw = df[TARGET_COL].values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

class_names = label_encoder.classes_
n_classes = len(class_names)

print(f"Samples: {len(y)}")
print(f"Classes: {class_names}")

# SCALE FEATURES

scaler = StandardScaler()
X = scaler.fit_transform(X)

# MODELS & HYPERPARAMETER GRIDS

models = {
    "DecisionTree": (
        DecisionTreeClassifier(random_state=SEED, class_weight="balanced"),
        {
            "max_depth": [None, 5, 10, 15],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "criterion": ["gini", "entropy", "log_loss"],
            "splitter": ["best", "random"]
        }
    ),
    "RandomForest": (
        RandomForestClassifier(random_state=SEED, class_weight="balanced"),
        {
            "n_estimators": [50, 100, 150],
            "max_depth": [None, 5, 10, 15],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["auto", "sqrt", "log2"],
            "criterion": ["gini", "entropy", "log_loss"]
        }
    ),
    "NaiveBayes": (
        GaussianNB(),
        {
            "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
        }
    ),
    "SVM": (
        SVC(probability=True, class_weight="balanced", random_state=SEED),
        {
            "C": [0.1, 1, 10],
            "kernel": ["linear", "rbf"],
            "gamma": ["scale", "auto"]
        }
    ),
    "XGBoost": (
        XGBClassifier(
            objective="multi:softprob" if n_classes > 2 else "binary:logistic",
            num_class=n_classes if n_classes > 2 else None,
            eval_metric="mlogloss",
            random_state=SEED,
            use_label_encoder=False
        ),
        {
            'n_estimators': [50, 100, 150],
            'learning_rate': [0.001, 0.01, 0.1, 0.5],
            'max_depth': [2, 3, 5, 7],
            'subsample': [0.8, 1.0],
            'colsample_bytree': [0.8, 1.0],
            'reg_lambda': [0, 1],
            'min_child_weight': [1, 3, 5]
        }
    )
}


# TRAIN & EVALUATE FUNCTION

def train_and_evaluate(name, model, param_grid, X, y):

    print(f"\n=== Training {name} ===")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    # Grid Search
    if param_grid:
        grid = GridSearchCV(
            model,
            param_grid,
            scoring="f1_macro",
            cv=skf,
            n_jobs=-1
        )
        grid.fit(X, y)
        best_model = grid.best_estimator_
        best_params = grid.best_params_
    else:
        best_model = model
        best_model.fit(X, y)
        best_params = {}

    y_true_all, y_pred_all, y_prob_all = [], [], []

    # Cross-validated predictions
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # XGBoost class safety
        if isinstance(best_model, XGBClassifier):
            missing = set(range(n_classes)) - set(np.unique(y_train))
            if missing:
                for m in missing:
                    X_train = np.vstack([X_train, X_train[0]])
                    y_train = np.append(y_train, m)

        best_model.fit(X_train, y_train)

        y_pred = best_model.predict(X_test)
        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)

        # Probabilities
        if hasattr(best_model, "predict_proba"):
            probs = best_model.predict_proba(X_test)
            if n_classes == 2 and probs.shape[1] == 1:
                probs = np.column_stack([1 - probs[:, 0], probs[:, 0]])
        else:
            probs = np.full((len(y_test), n_classes), np.nan)

        y_prob_all.append(probs)

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.vstack(y_prob_all)


    # METRICS
   
    acc = accuracy_score(y_true_all, y_pred_all)
    prec = precision_score(y_true_all, y_pred_all, average="macro", zero_division=0)
    rec = recall_score(y_true_all, y_pred_all, average="macro", zero_division=0)
    f1 = f1_score(y_true_all, y_pred_all, average="macro", zero_division=0)

    print(classification_report(y_true_all, y_pred_all, target_names=class_names))


    # CONFUSION MATRIX
  
    cm = confusion_matrix(y_true_all, y_pred_all)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names,
                yticklabels=class_names)
    plt.title(f"{name} – Confusion Matrix")
    plt.savefig(os.path.join(OUTPUT_DIR, f"{name}_CM.png"))
    plt.close()


    # ROC CURVE
 
    plt.figure(figsize=(7,6))

    if n_classes == 2:
        y_score = y_prob_all[:, 1]
        fpr, tpr, _ = roc_curve(y_true_all, y_score)
        roc_auc_val = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"AUC = {roc_auc_val:.2f}")
    else:
        y_bin = label_binarize(y_true_all, classes=range(n_classes))
        roc_auc_val = 0
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob_all[:, i])
            auc_i = auc(fpr, tpr)
            roc_auc_val += auc_i
            plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC={auc_i:.2f})")
        roc_auc_val /= n_classes

    plt.plot([0,1], [0,1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{name} – ROC Curve")
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{name}_ROC.png"))
    plt.close()

    return {
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": roc_auc_val,
        "Best_Params": best_params
    }


# RUN ALL MODELS

results = []
for name, (model, params) in models.items():
    results.append(train_and_evaluate(name, model, params, X, y))

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUTPUT_DIR, "GLCM_Final_ML_Results.csv"), index=False)
print("\nFINAL RESULTS:")
print(results_df)
